### IMPORT

In [1]:
from pathlib import Path

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
import rootutils

import sys
rootutils.setup_root(Path(".").resolve(), indicator=".project-root", pythonpath=True)
sys.path.append(str(Path(".").resolve().parent / "src"))

from bev.models.bev_emb import BEVAdapterViT
from bev.models.bevqa import BEVQA_ViT
from bev.data.bevqa_dataset import BEVQADataset
from bev.models.head import OutputHead
from bev.models.mca import MCALayer
from bev.models.text_emb import TextAdapter
from bev.training.train import train_epoch, val_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"

### PATH

In [2]:
import hydra
from hydra import initialize, compose
from omegaconf import OmegaConf

try:
    initialize(config_path="../configs", version_base=None)
except:
    pass

cfg = compose(config_name="config")

FEAT_DIR = Path(cfg.paths.bev_features_dir)
DICT_DIR = Path(cfg.paths.dict_dir)
GLOVE = Path(cfg.paths.glove_path)

### DATASET

In [3]:
train_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "train",
    json_path=DICT_DIR / "NuScenes_train_questions.json",
    glove=GLOVE
)
val_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "val",
    json_path=DICT_DIR / "NuScenes_val_questions.json",
    glove=GLOVE
)

30
30


In [4]:
feat, quest, ans = train_dataset[0]
print(feat.shape, quest.shape, ans)

torch.Size([128, 200, 200]) torch.Size([30, 300]) tensor(17)


In [10]:
print(len(train_dataset), len(val_dataset))

376604 83337


### DATALOADER

In [11]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [12]:
feat, quest, ans = next(iter(train_dataloader))
print(feat.shape, quest.shape, ans)

torch.Size([32, 128, 200, 200]) torch.Size([32, 30, 300]) tensor([18, 20, 28, 25, 17, 21, 17, 18,  6, 15, 29, 29, 18,  2,  8, 16, 29, 29,
        14, 21, 21, 18, 14, 29, 29, 29, 14, 29, 28, 18, 20, 18])


### EMBEDDINGS

In [ ]:
bev_ad = BEVAdapterViT() # BEV MAP -> BEV EMB
text_ad = TextAdapter() # [B,32,512] to match BEV emb [B,400,512] 
feat_output = bev_ad(feat)
text_output = text_ad(quest)
print(f"Feat:{feat_output.shape}") # [B,100,512]
print(f"Text:{text_output.shape}") # [B,32,512]

Feat:torch.Size([32, 100, 512])
Text:torch.Size([32, 30, 512])


### MODEL

In [14]:
model = BEVQA_ViT()
out = model(feat, quest)
print(out.shape) # [4, 30]

torch.Size([32, 30])


### TRAIN

In [ ]:
model = BEVQA_ViT().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 20

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_epoch(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_dataloader, criterion, device)
    print(f"Epoch {epoch+1:02d}/20 | Train Loss: {tr_loss:.4f} - Acc: {tr_acc:.4f} | Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), ROOT / "best_model_vit.pth")
        print(f"  → Saved best model (val_acc={val_acc:.4f})")